# Module 3b: final scoring (test split and own recordings, ONCE)
Run only after `gate.threshold` is set and committed, and after `python -m readback_guard.batch test` and `... batch own`.

In [1]:
import subprocess
import pandas as pd
from readback_guard.config import ROOT, load_config
from readback_guard.asr import load_asr_cases, load_own_cases, wer_table, critical_values
from readback_guard.evaluate import load_cases, rates, log_result, attribution, mcnemar, md_table
from readback_guard.score import run_cases

cfg = load_config()
t = cfg["gate"]["threshold"]
assert t is not None, "set gate.threshold on dev first"
dirty = subprocess.run(["git", "status", "--porcelain", "labels", "configs"], cwd=ROOT, capture_output=True, text=True).stdout
assert not dirty, "commit labels/ and configs/ before scoring"
t

0.4

In [2]:
test = load_cases(cfg, "test")
test_real = test.query("kind == 'real'")
test_asr = load_asr_cases(cfg, "test")

runs = {
    "1b human text, real + injected": run_cases(test, cfg),
    "human text, real pairs": run_cases(test_real, cfg),
    "Whisper text, real pairs": run_cases(test_asr, cfg),
    "Whisper + gate, real pairs": run_cases(test_asr, cfg, t),
}
main = pd.DataFrame({name: rates(df) for name, df in runs.items()}).T
main

,n_errors,detected,detection_rate,detection_ci,n_correct,false_alarms,false_alarm_rate,false_alarm_ci,exact_verdict_rate,say_again,say_again_rate,false_accusations,missed
"1b human text, real + injected",106,100,0.943,"(0.882, 0.974)",82,17,0.207,"(0.134, 0.307)",0.814,0,0.0,17,6
"human text, real pairs",24,23,0.958,"(0.798, 0.993)",82,17,0.207,"(0.134, 0.307)",0.774,0,0.0,17,1
"Whisper text, real pairs",24,24,1.0,"(0.862, 1.0)",82,51,0.622,"(0.514, 0.719)",0.415,0,0.0,51,0
"Whisper + gate, real pairs",24,24,1.0,"(0.862, 1.0)",82,56,0.683,"(0.576, 0.774)",0.34,19,0.179,45,0


In [3]:
for name, df in runs.items():
    log_result("final_" + name.split(",")[0].replace(" ", "_").replace("+", "plus"), "test",
               "asr" if "Whisper" in name else "ref", rates(df), cfg, notes=name)

In [4]:
wer = wer_table(test_asr)
cv = critical_values(test_asr)
print("WER", wer, " critical-value error rate", round(1 - cv.asr_ok.mean(), 3), "of", len(cv))
cv.groupby(["side", "type"]).asr_ok.agg(["size", "mean"]).round(2)

WER {'atco': 0.174, 'pilot': 0.385, 'all': 0.25}  critical-value error rate 0.161 of 366


size  mean
side  type                 
atco  callsign    100  0.90
      frequency    18  0.61
      runway       79  1.00
      speed         2  0.50
      squawk        6  1.00
pilot callsign     81  0.69
      frequency    12  0.42
      runway       65  0.86
      squawk        3  1.00

In [5]:
m, table = attribution(runs["human text, real pairs"], runs["Whisper text, real pairs"])
mg, table_g = attribution(runs["human text, real pairs"], runs["Whisper + gate, real pairs"])
print("human vs Whisper:", mcnemar(m))
print("human vs Whisper + gate:", mcnemar(mg))
table

human vs Whisper: {'human_only_right': 40, 'asr_only_right': 7, 'p_value': 0.0}
human vs Whisper + gate: {'human_only_right': 43, 'asr_only_right': 5, 'p_value': 0.0}


cause,language understanding,right on both,right only on Whisper text,speech recognition,total
gold,,,,,
correct readback (false alarm if wrong),11,25,6,40,82
real error (miss if wrong),0,23,1,0,24
total,11,48,7,40,106


## Own recordings (real speech with real errors)

In [6]:
own_human = run_cases(load_own_cases(cfg, "human"), cfg)
own_asr_cases = load_own_cases(cfg, "asr")
own_asr = run_cases(own_asr_cases, cfg)
own_gate = run_cases(own_asr_cases, cfg, t)
own = pd.DataFrame({"script text": rates(own_human), "Whisper text": rates(own_asr), "Whisper + gate": rates(own_gate)}).T
for name, df in [("own_script", own_human), ("own_whisper", own_asr), ("own_whisper_gate", own_gate)]:
    log_result(name, "own", "asr" if "whisper" in name else "ref", rates(df), cfg)
own

,n_errors,detected,detection_rate,detection_ci,n_correct,false_alarms,false_alarm_rate,false_alarm_ci,exact_verdict_rate,say_again,say_again_rate,false_accusations,missed
script text,10,10,1.0,"(0.722, 1.0)",10,0,0.0,"(-0.0, 0.278)",1.0,0,0.0,0,0
Whisper text,10,10,1.0,"(0.722, 1.0)",10,4,0.4,"(0.168, 0.687)",0.7,0,0.0,4,0
Whisper + gate,10,10,1.0,"(0.722, 1.0)",10,4,0.4,"(0.168, 0.687)",0.7,0,0.0,4,0


In [7]:
mo, table_o = attribution(own_human, own_asr)
print(mcnemar(mo))
per_rec = own_human[["case_id", "verdict", "pred"]].rename(columns={"pred": "script"})
per_rec["whisper"], per_rec["whisper_gate"], per_rec["reason"] = own_asr.pred, own_gate.pred, own_gate.reason
per_rec

{'human_only_right': 4, 'asr_only_right': 0, 'p_value': 0.125}


,case_id,verdict,script,whisper,whisper_gate,reason
0,r01,CORRECT,CORRECT,CORRECT,CORRECT,all items read back
1,r02,WRONG_VALUE,WRONG_VALUE,WRONG_CALLSIGN,WRONG_CALLSIGN,callsign sunza 452 read back as lufthansa 452
2,r03,CORRECT,CORRECT,WRONG_CALLSIGN,WRONG_CALLSIGN,callsign liner 3ab read back as ryanair 3ab
3,r04,MISSING_ITEM,MISSING_ITEM,MISSING_ITEM,MISSING_ITEM,frequency 127.125 not read back
4,r05,CORRECT,CORRECT,CORRECT,CORRECT,all items read back
5,r06,WRONG_CALLSIGN,WRONG_CALLSIGN,WRONG_CALLSIGN,WRONG_CALLSIGN,callsign visor 530 read back as vizir 538
6,r07,CORRECT,CORRECT,WRONG_CALLSIGN,WRONG_CALLSIGN,callsign str6ma read back as austrian 6ma
7,r08,WRONG_VALUE,WRONG_VALUE,WRONG_VALUE,WRONG_VALUE,squawk 2351 read back as 2357
8,r09,CORRECT,CORRECT,CORRECT,CORRECT,all items read back
9,r10,WRONG_VALUE,WRONG_VALUE,WRONG_CALLSIGN,WRONG_CALLSIGN,callsign liner 82k read back as ryanair 82k


In [8]:
cols = ["n_errors", "detected", "detection_rate", "detection_ci", "n_correct", "false_alarms",
        "false_alarm_rate", "false_alarm_ci", "false_accusations", "missed", "say_again", "say_again_rate"]
parts = ["## Test split", md_table(main[cols]), f"WER {wer}, critical-value error rate {1 - cv.asr_ok.mean():.3f} (n={len(cv)})",
         "## Attribution, human vs Whisper (test, real pairs)", md_table(table), str(mcnemar(m)),
         "## Attribution, human vs Whisper + gate", md_table(table_g), str(mcnemar(mg)),
         "## Own recordings", md_table(own[cols]), md_table(per_rec.set_index("case_id"))]
(ROOT / "results" / "tables.md").write_text("\n\n".join(parts))
print("wrote results/tables.md")

wrote results/tables.md
